# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR² clinical oncology dataset using the `mlcroissant` library, referencing all data entities (record sets, fields, columns) by their `@id` attributes as required by the Croissant standard. The steps are based on the official `mlcroissant` example notebooks and are structured for reproducible and FAIR analysis.

### Dataset Source
The dataset Croissant schema is available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Initialize the Dataset
dataset = mlc.Dataset(croissant_url)

# Access and display metadata (as an object, not subscripting)
meta = dataset.metadata
print(f"Dataset name: {meta.name}\n")
print(f"Dataset description: {meta.description}\n")
print(f"Published: {meta.datePublished}  |  Version: {meta.version}\n")
print(f"License: {meta.license}\n")

## 2. Data Overview

Review available record sets and their contents by listing their `@id` and associated field IDs. All references follow the Croissant `@id` convention.

In [ ]:
# List all record sets with their @id and the fields they contain
record_sets = dataset.record_sets  # List of RecordSet objects
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            field_ids = [field.id for field in rs.fields]
            print(f"  Fields: {field_ids}")
        else:
            print("  No fields available.")
        print("")

Let's look at the records from the main tabular RecordSet by its `@id`.

In [ ]:
# Select the tabular medical RecordSet @id
# For this dataset, the primary data RecordSet is usually suffixed or named with clinical/medical concepts
# Here, let's fetch the first found RecordSet for illustration:
if not record_sets:
    print("No record sets available to load records.")
else:
    primary_record_set_id = record_sets[0].id
    print(f"Showing a few records from RecordSet @id: {primary_record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(rec)
        if i >= 2:
            print("...\n(Showing only the first 3 records)")
            break

## 3. Data Extraction

Load all available record sets into pandas DataFrames using only `@id` references for record sets and fields.

In [ ]:
# Prepare the list of RecordSet @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

dfs = {}
for record_set_id in record_set_ids:
    # List of dicts
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)

# Show columns of the primary record set
primary_df = dfs[record_set_ids[0]]
print(f"Columns from RecordSet @id {record_set_ids[0]}:")
print(list(primary_df.columns))

# Show first 5 records
primary_df.head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing using field `@id` references (not by column labels directly), focusing on numeric and categorical variables.

In [ ]:
# Examine available fields in the primary RecordSet
primary_fields = dataset.record_sets[0].fields
print("Available field @ids in the primary RecordSet:")
for field in primary_fields:
    print(f"  - {field.id} | name: {getattr(field, 'name', '<no-name>')}")

# Example: Choose a numeric field @id (assuming 'age' is present)
# You may replace this with the actual numeric field @id as printed above if it's different
numeric_field_id = None
for field in primary_fields:
    field_id = field.id
    if hasattr(field, 'data_type') and field.data_type in ['Integer', 'Float', 'Number', 'schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_field_id = field_id
        print(f"Selected numeric field @id: {numeric_field_id}")
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found in primary RecordSet.")

# Filtering example: Show records with numeric field > 50 (e.g. age > 50)
threshold = 50
filtered = primary_df[primary_df[numeric_field_id].astype(float) > threshold]
print(f"Number of records with {numeric_field_id} > {threshold}: {len(filtered)}\n")
print(filtered[[numeric_field_id]].head())

# Normalize the numeric field
if not filtered.empty:
    filtered[numeric_field_id + "_normalized"] = (filtered[numeric_field_id].astype(float) - filtered[numeric_field_id].astype(float).mean()) / filtered[numeric_field_id].astype(float).std()
    print(f"Normalized values of {numeric_field_id} (first five records):")
    print(filtered[[numeric_field_id, numeric_field_id + "_normalized"]].head())

# Find a likely grouping categorical field (e.g. sex, anatomical location, msi_status)
group_field_id = None
for field in primary_fields:
    if hasattr(field, 'data_type') and field.data_type == 'Text' and field.id != numeric_field_id:
        group_field_id = field.id
        print(f"Selected group-by field @id: {group_field_id}")
        break

if group_field_id and group_field_id in filtered.columns:
    grouped = filtered.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped)


## 5. Visualization

Visualize distributions or relationships in the dataset using matplotlib and pandas, always using field `@id`s for column references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field (e.g. age) in the primary record set
plt.figure(figsize=(7, 5))
sns.histplot(primary_df[numeric_field_id].astype(float), kde=True, bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If a group field is available, plot boxplot
if group_field_id and group_field_id in primary_df.columns:
    plt.figure(figsize=(9, 5))
    sns.boxplot(x=primary_df[group_field_id], y=primary_df[numeric_field_id].astype(float))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- This notebook demonstrated how to programmatically load, explore, and visualize a FAIR² clinical oncology dataset using Croissant and mlcroissant.
- All data access (record sets and fields) was done using only `@id` references, ensuring full traceability.
- You may extend this notebook to perform further statistical analyses, domain-specific feature engineering, and machine learning as required.
